# 03 — Démonstration d'inférence sur les 4 exemples de référence

Charge le modèle entraîné depuis `data/models/` et confronte les prédictions aux résultats attendus (✓ / ✗) avec les scores de confiance.

**Prérequis** : modèles entraînés (`python scripts/run_training.py`).


In [ ]:
import sys, os
# Se placer à la racine du projet (le notebook est dans notebooks/).
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print("Racine projet :", ROOT)


In [ ]:
from src.utils import load_config
from src.inference.predictor import VerbatimPredictor
cfg = load_config()
predictor = VerbatimPredictor(cfg)  # charge niv1, niv2, sentiment, signaux
print('Modèles chargés.')


In [ ]:
EXEMPLES = [
    {
        "verbatim": ("Plus de 2 semaines de retard. Sans notification pour prévenir "
                     "que le délai d'une semaine était décalé à plus de 2 semaines."),
        "satisfaction": 2,
        "attendu": {"theme1_niv1": "Suivi de commande et livraison",
                     "theme1_niv2": "Délai non respecté", "theme1_sentiment": "Négatif",
                     "signal_rupture_client": False, "signal_churn": True,
                     "signal_insatisfaction_forte": True},
    },
    {
        "verbatim": ("Bug au moment du paiement et ensuite produit indiqué en stock "
                     "et finalement commande annulée car rupture."),
        "satisfaction": 2,
        "attendu": {"theme1_niv1": "Tunnel de vente - Paiement", "theme1_niv2": "Bug paiement",
                     "theme2_niv1": "Annulation commande",
                     "theme2_niv2": "Produit disponible devenu en rupture",
                     "signal_churn": True},
    },
    {
        "verbatim": ("J'ai perdu mes points et pas de bon d'achat, "
                     "je ne suis pas près de racheter chez Cultura."),
        "satisfaction": 2,
        "attendu": {"theme1_niv1": "Programme de fidélité", "theme1_niv2": "Bon d'achat non reçu",
                     "theme1_sentiment": "Négatif", "signal_rupture_client": True,
                     "signal_churn": True},
    },
    {
        "verbatim": "Accès facile au produit souhaité. Navigation très intuitive.",
        "satisfaction": 9,
        "attendu": {"theme1_niv1": "Trouver son produit", "theme1_niv2": "Facile",
                     "theme1_sentiment": "Positif", "signal_rupture_client": False,
                     "signal_churn": False},
    },
]


## Prédictions vs attendus


In [ ]:
import pandas as pd
def check(pred, attendu):
    oks = {}
    for k,v in attendu.items():
        oks[k] = (pred.get(k) == v)
    return oks
rows = []
for ex in EXAMPLES:
    pred = predictor.predict(ex['verbatim'], satisfaction_score=ex['satisfaction'])
    oks = check(pred, ex['attendu'])
    rows.append({
        'verbatim': ex['verbatim'][:55]+'...',
        'niv1_pred': pred['theme1_niv1'], 'niv1_ok': '✓' if oks.get('theme1_niv1',True) else '✗',
        'niv2_pred': pred['theme1_niv2'], 'niv2_ok': '✓' if oks.get('theme1_niv2',True) else '✗',
        'sentiment': pred['theme1_sentiment'],
        'rupture': pred['signal_rupture_client'], 'churn': pred['signal_churn'],
        'confiance': pred['confidence_globale'],
        'tout_ok': '✓' if all(oks.values()) else '✗',
    })
pd.DataFrame(rows)


## Détail complet d'une prédiction


In [ ]:
import json
print(json.dumps(predictor.predict(EXAMPLES[1]['verbatim'], satisfaction_score=2), ensure_ascii=False, indent=2))
